# Segmentacion de instancias: decoder semantico, centros y offsets

Define el decoder de segmentacion con skip connections, sus objetivos y perdidas, el postprocesamiento a IDs de fragmento, las metricas de segmentacion, la restauracion a la cuadricula del MHA, la asociacion 3D entre cortes y el adaptador de SAM. Todas las pruebas son estructurales u oraculo: el decoder no se entrena aqui (eso ocurre en Entrenamiento_multitarea).

**Entradas:** `splits/dataset.csv`, el manifiesto de cortes positivos, el backbone (`Backbone`), el dataset (`Targets_deteccion`), `deteccion_grid.py` y `cbam_modulos.py`.

Esta sección implementa una propuesta propia para separar fragmentos: el decoder predice la región anatómica de cada píxel, centros de fragmento por región y vectores (offsets) desde los píxeles óseos hacia el centro de su instancia. Un postprocesamiento combina esas señales y devuelve IDs por fragmento.

### ¿Por qué elegimos centros + offsets?

La máscara semántica identifica sacro, coxal izquierdo y coxal derecho, pero por sí sola asignaría la misma etiqueta a todos los fragmentos de una región y no conservaría su identidad individual. La consigna del avance requiere separar cada fragmento, incluso cuando hay varios de la misma región. Por eso añadimos centros y offsets: cada centro propone una instancia y los píxeles óseos votan hacia el centro de su propio fragmento. Así podemos manejar un número variable de instancias sin reservar un canal fijo para cada ID, que cambia entre cortes y pacientes.

También se consideraron componentes conectados y bordes. Los componentes conectados pueden fusionar fragmentos que se tocan; una señal de borde es otra alternativa válida, pero requiere decidir cómo cerrar contornos incompletos. Centros + offsets ofrece una señal explícita de agrupación y un postprocesamiento determinista para reconstruir IDs. Es una decisión de diseño del equipo, no una arquitectura exigida literalmente por Ferro; se debe verificar con fragmentos pequeños y en contacto, usando `val`.

Las máscaras objetivo provienen de los MHA reales y de los splits fijos. Esta es una implementación inicial; no sustituye la integración ni el entrenamiento multitarea completo, y sus parámetros deben seleccionarse con validación, nunca con test.

## 1. Configuracion

Rutas relativas a la raiz del repositorio y lectura de los archivos que generan los notebooks anteriores. Las definiciones compartidas se cargan por nombre desde el notebook donde viven (`definiciones_notebooks.cargar_definiciones`), sin ejecutar sus celdas de prueba.

In [1]:
from pathlib import Path
import copy
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import SimpleITK as sitk
import torch
import torch.nn as nn
import torch.nn.functional as F

# Raiz del repositorio: funciona si el kernel arranca en notebooks/ o en la raiz.
BASE_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))  # para importar los modulos de src/

from src.definiciones_notebooks import cargar_definiciones

SPLITS_DIR = BASE_DIR / "splits"
from src.rutas import MANIFEST_CSV
NOTEBOOKS_DIR = BASE_DIR / "notebooks"

def to_repo_relative(path):
    """Guarda rutas portables dentro del repositorio."""
    path = Path(path)
    try:
        return path.relative_to(BASE_DIR).as_posix()
    except ValueError:
        return path.as_posix()


def resolve_repo_path(path):
    """Permite leer rutas relativas o absolutas sin romper compatibilidad."""
    path = Path(path)
    if path.is_absolute():
        return path
    return BASE_DIR / path

# Dataset curado y splits generados por Dataset_splits.ipynb
df = pd.read_csv(SPLITS_DIR / "dataset.csv", dtype={"case_id": str})
# Indice de cortes positivos generado por Targets_deteccion.ipynb
slice_manifest = pd.read_csv(MANIFEST_CSV, dtype={"case_id": str})

cargar_definiciones(NOTEBOOKS_DIR / "Ventaneo_hu.ipynb", {"load_mha", "apply_hu_window", "enhance_bone_contrast"}, destino=globals())
cargar_definiciones(NOTEBOOKS_DIR / "Targets_deteccion.ipynb", {"instance_to_semantic", "boxes_from_semantic", "letterbox_pair", "build_detection_target",
     "PengwinDetectionDataset", "detection_collate"},
                    valores={"CLASS_ID_TO_NAME", "TARGET_COLORS"}, destino=globals())
cargar_definiciones(NOTEBOOKS_DIR / "Backbone.ipynb", {"FundidoraPC", "FundidoraPCPlus", "build_backbone", "theoretical_receptive_field",
     "count_parameters", "make_models"}, destino=globals())

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NAMES = ["base", "plus"]
MODELS = make_models()          # backbones sin entrenar, con la misma semilla que en Backbone
print("torch", torch.__version__, "| dispositivo:", DEVICE)

torch 2.11.0+cu128 | dispositivo: cuda


## 2. Crear objetivos de centros y offsets desde las máscaras reales

Para cada píxel se conserva su clase anatómica y su ID de instancia. El centroide de cada ID genera un mapa de centros gaussiano; los offsets indican cuánto debe desplazarse cada píxel para llegar a ese centro. Se usa el ID original solo para crear la supervisión, no como una clase fija que el modelo deba memorizar entre pacientes.

In [2]:
def backbone_feature_pyramid(backbone, x):
   
    f1 = backbone.pool1(torch.relu(backbone.bn1(backbone.conv1(x))))   # [B, 32, H/2,  W/2 ]
    f2 = backbone.pool2(torch.relu(backbone.bn2(backbone.conv2(f1))))  # [B, 64, H/4,  W/4 ]
    f3 = backbone.pool3(torch.relu(backbone.bn3(backbone.conv3(f2))))  # [B,128, H/8,  W/8 ]
    f4 = backbone.pool4(torch.relu(backbone.bn4(backbone.conv4(f3))))  # [B,256, H/16, W/16]
    if hasattr(backbone, "context"):
        f4 = f4 + backbone.context(f4)
    return [f1, f2, f3, f4]

In [3]:
def build_instance_segmentation_targets(instance_mask, sigma=3.0, center_mode="interior"):
    """center_mode="mean": centro = promedio de los píxeles del fragmento (versión original).
    center_mode="interior": igual, salvo cuando el promedio cae FUERA del fragmento (sección en forma de C o en
    varios trozos): entonces se usa el punto más interior del trozo más grande. Así el pico de centro que aprende
    el decoder siempre queda sobre el hueso y un mismo fragmento no reparte sus votos hacia el vacío."""
    from scipy import ndimage
    instance_mask = np.asarray(instance_mask, dtype=np.int32)
    if instance_mask.ndim != 2:
        raise ValueError(f"Se esperaba máscara 2D (y, x), se recibió {instance_mask.shape}")
    if sigma <= 0:
        raise ValueError("sigma debe ser positivo")
    if center_mode not in ("mean", "interior"):
        raise ValueError("center_mode debe ser 'mean' o 'interior'")

    height, width = instance_mask.shape
    semantic = instance_to_semantic(instance_mask).astype(np.int64)
    center_heatmaps = np.zeros((3, height, width), dtype=np.float32)
    offsets = np.zeros((2, height, width), dtype=np.float32)
    foreground = instance_mask > 0
    yy, xx = np.ogrid[:height, :width]
    radius = int(np.ceil(3 * sigma))

    for instance_id in np.unique(instance_mask[foreground]):
        instance_id = int(instance_id)
        if 1 <= instance_id <= 10:
            class_id = 1
        elif 11 <= instance_id <= 20:
            class_id = 2
        elif 21 <= instance_id <= 30:
            class_id = 3
        else:
            raise ValueError(f"ID de fragmento fuera de 1..30: {instance_id}")

        ys, xs = np.nonzero(instance_mask == instance_id)
        center_y, center_x = float(ys.mean()), float(xs.mean())
        if center_mode == "interior" and instance_mask[int(round(center_y)), int(round(center_x))] != instance_id:
            top, left = int(ys.min()), int(xs.min())
            crop = instance_mask[top:int(ys.max()) + 1, left:int(xs.max()) + 1] == instance_id
            parts, n_parts = ndimage.label(crop)
            if n_parts > 1:                                   # solo el trozo más grande
                crop = parts == int(np.bincount(parts.ravel())[1:].argmax()) + 1
            depth = ndimage.distance_transform_edt(np.pad(crop, 1))[1:-1, 1:-1]
            iy, ix = np.unravel_index(int(depth.argmax()), depth.shape)
            center_y, center_x = float(top + iy), float(left + ix)
        y0, y1 = max(0, int(center_y) - radius), min(height, int(center_y) + radius + 1)
        x0, x1 = max(0, int(center_x) - radius), min(width, int(center_x) + radius + 1)
        gy, gx = np.mgrid[y0:y1, x0:x1]
        gaussian = np.exp(-((gy - center_y) ** 2 + (gx - center_x) ** 2) / (2 * sigma**2))
        gaussian /= max(float(gaussian.max()), 1e-8)
        center_heatmaps[class_id - 1, y0:y1, x0:x1] = np.maximum(
            center_heatmaps[class_id - 1, y0:y1, x0:x1], gaussian
        )
        offsets[0, ys, xs] = (center_x - xs) / width
        offsets[1, ys, xs] = (center_y - ys) / height

    return {
        "semantic": semantic,
        "centers": center_heatmaps,
        "offsets": offsets,
        "foreground": foreground,
        "instance_ids": np.unique(instance_mask[foreground]).astype(np.int32),
    }


## 3. Decoder de segmentación sobre el mapa compartido

El decoder recibe el mapa espacial `[B, 256, 16, 16]` del backbone/atención y lo aumenta progresivamente hasta la resolución de la máscara `[B, *, 256, 256]`. Además recibe priors espaciales de las cajas por región: se construyen con cajas reales durante entrenamiento y con cajas predichas después de NMS durante inferencia. Devuelve logits semánticos (4 canales), centros (3 canales, uno por región) y offsets `(x, y)`. Las ramas comparten las características decodificadas. El helper acepta el mismo contrato `{boxes, labels}` para cualquiera de las dos fuentes; no usa cajas reales durante inferencia. Como las cajas anotadas son más precisas que las predichas, al integrar conviene evaluar jitter de cajas o una transición gradual a cajas predichas durante el entrenamiento; la prueba con cajas reales solo verifica formas y gradientes, no elimina esa diferencia. Se usa interpolación bilineal seguida de convoluciones, en vez de convoluciones transpuestas, para evitar la configuración de kernel/stride que las guías advierten que puede causar artefactos de cuadrícula.

**Skip connections (ya implementadas).** Este bloque define el decoder original, que solo recibe el mapa de 16 × 16 y se conserva únicamente como línea base sin skips para la ablación. El decoder que se usa en la integración y en el entrenamiento es `CenterOffsetSkipDecoder` (siguiente celda de código): concatena mapas intermedios del backbone (stride 8, 4 y 2) y una rama de la imagen a resolución completa, y alinea los canales con una convolución 1 × 1 después de cada concatenación. Con `use_skips=False` el mismo decoder queda como baseline tipo FCN, con idéntica profundidad y anchos, para comparar con/sin skips en igualdad de condiciones. El backbone compartido no se modifica.

In [4]:
def build_detection_region_priors(detections, output_size=(256, 256), device=None):
    """Rasteriza cajas xyxy/clases 1..3 como tres priors espaciales por imagen."""
    height, width = map(int, output_size)
    if height <= 0 or width <= 0:
        raise ValueError("output_size debe tener dimensiones positivas")
    if isinstance(detections, dict):
        detections = [detections]

    priors = torch.zeros((len(detections), 3, height, width), dtype=torch.float32, device=device)
    for batch_index, detection in enumerate(detections):
        boxes = torch.as_tensor(detection["boxes"]).detach().to(device="cpu", dtype=torch.float32).numpy()
        labels = torch.as_tensor(detection["labels"]).detach().to(device="cpu", dtype=torch.int64).numpy()
        boxes = boxes.reshape(-1, 4)
        labels = labels.reshape(-1)
        if len(boxes) != len(labels):
            raise ValueError("Debe existir una clase por cada caja")
        for box, class_id in zip(boxes, labels):
            class_id = int(class_id)
            if class_id not in (1, 2, 3):
                raise ValueError(f"Clase anatómica fuera de 1..3: {class_id}")
            x0 = int(np.clip(np.floor(box[0]), 0, width))
            y0 = int(np.clip(np.floor(box[1]), 0, height))
            x1 = int(np.clip(np.ceil(box[2]), 0, width))
            y1 = int(np.clip(np.ceil(box[3]), 0, height))
            if x1 > x0 and y1 > y0:
                priors[batch_index, class_id - 1, y0:y1, x0:x1] = 1.0
    return priors


class SegmentationConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.layers(x)


In [5]:
class SkipFusionBlock(nn.Module):
    """Sube el decoder, concatena la skip, ALINEA canales con 1x1 y refina con 2 convs 3x3."""
    def __init__(self, in_channels, skip_channels, out_channels, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        fused = in_channels + (skip_channels if use_skip else 0)
        self.align = nn.Sequential(                      # 1x1 tras concatenar: evita explosión de canales
            nn.Conv2d(fused, out_channels, kernel_size=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.GELU(),
        )
        self.refine = SegmentationConvBlock(out_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        if self.use_skip:
            x = torch.cat([x, skip], dim=1)
        return self.refine(self.align(x))


class CenterOffsetSkipDecoder(nn.Module):
    """Mismo contrato de salida que CenterOffsetSegmentationDecoder, pero con skips del encoder
    (f3, f2, f1) y una rama stem a resolución completa. `use_skips=False` da el baseline tipo FCN
    con la MISMA profundidad y anchos (para el experimento controlado con/sin skips)."""
    def __init__(self, in_channels=256, skip_channels=(128, 64, 32), image_channels=1,
                 stem_channels=16, out_channels=(128, 96, 64, 32), use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        self.stem = nn.Sequential(
            nn.Conv2d(image_channels, stem_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, stem_channels),
            nn.GELU(),
        )
        skip_chs = tuple(skip_channels) + (stem_channels,)          # f3, f2, f1, stem
        blocks, current = [], in_channels
        for skip_c, out_c in zip(skip_chs, out_channels):
            blocks.append(SkipFusionBlock(current, skip_c, out_c, use_skip=use_skips))
            current = out_c
        self.blocks = nn.ModuleList(blocks)
        self.semantic_head = nn.Conv2d(current + 3, 4, kernel_size=1)
        self.center_head = nn.Conv2d(current + 3, 3, kernel_size=1)
        self.offset_head = nn.Conv2d(current + 3, 2, kernel_size=1)

    def forward(self, image, features, region_priors, output_size=(256, 256)):
        f1, f2, f3, f4 = features
        if region_priors is None or region_priors.ndim != 4 or region_priors.shape[:2] != (f4.shape[0], 3):
            raise ValueError("region_priors debe tener forma [B, 3, H, W] y corresponder a las cajas")
        skips = [f3, f2, f1, self.stem(image)]
        x = f4
        for block, skip in zip(self.blocks, skips):
            x = block(x, skip)
        if tuple(x.shape[-2:]) != tuple(output_size):
            x = F.interpolate(x, size=output_size, mode="bilinear", align_corners=False)
        if tuple(region_priors.shape[-2:]) != tuple(output_size):
            region_priors = F.interpolate(region_priors.float(), size=output_size, mode="nearest")
        x = torch.cat([x, region_priors.to(device=x.device, dtype=x.dtype)], dim=1)
        return {
            "semantic_logits": self.semantic_head(x),
            "center_logits": self.center_head(x),
            "offsets": torch.tanh(self.offset_head(x)),
        }


def run_skip_segmentation(backbone, decoder, image, region_priors, cbam=None, output_size=(256, 256)):
    """Backbone -> (CBAM opcional sobre f4) -> decoder con skips. No modifica el backbone."""
    features = backbone_feature_pyramid(backbone, image)
    if cbam is not None:
        refined = cbam(features[-1])
        features[-1] = refined[0] if isinstance(refined, tuple) else refined
    return decoder(image, features, region_priors, output_size=output_size)

In [6]:
_x = torch.rand(2, 1, 256, 256, device=DEVICE)
_pri = torch.zeros(2, 3, 256, 256, device=DEVICE)
_pri[:, 0, 40:120, 40:120] = 1.0

for _name in NAMES:
    _bb = MODELS[_name].eval()
    with torch.no_grad():
        _feats = backbone_feature_pyramid(_bb, _x)
        _ref = _bb._encode(_x)
    assert [tuple(f.shape[1:]) for f in _feats] == [(32, 128, 128), (64, 64, 64), (128, 32, 32), (256, 16, 16)]
    assert torch.allclose(_feats[-1], _ref, atol=1e-6), f"{_name}: f4 debe ser igual a _encode"
    print(f"[{_name}] pirámide OK y f4 == _encode")

# Contrato de salida y conteo de parámetros (con y sin skips)
for _use in (True, False):
    _dec = CenterOffsetSkipDecoder(in_channels=256, image_channels=1, use_skips=_use).to(DEVICE).eval()
    with torch.no_grad():
        _out = run_skip_segmentation(MODELS["base"], _dec, _x, _pri)
    assert _out["semantic_logits"].shape == (2, 4, 256, 256)
    assert _out["center_logits"].shape == (2, 3, 256, 256)
    assert _out["offsets"].shape == (2, 2, 256, 256)
    print(f"use_skips={_use}: salida OK | parámetros del decoder = {sum(p.numel() for p in _dec.parameters()):,}")

# La skip realmente se usa: perturbar f1 debe cambiar la salida solo con skips
for _use in (True, False):
    torch.manual_seed(SEED)
    _dec = CenterOffsetSkipDecoder(use_skips=_use).to(DEVICE).eval()
    with torch.no_grad():
        _f = backbone_feature_pyramid(MODELS["base"], _x)
        _a = _dec(_x, _f, _pri)["semantic_logits"]
        _f2 = [t.clone() for t in _f]; _f2[0] = _f2[0] + 1.0
        _b = _dec(_x, _f2, _pri)["semantic_logits"]
    _delta = (_a - _b).abs().max().item()
    assert (_delta > 1e-6) == _use, (_use, _delta)
    print(f"use_skips={_use}: cambio de la salida al perturbar f1 = {_delta:.3e}")
print("Skip connections verificadas.")

[base] pirámide OK y f4 == _encode
[plus] pirámide OK y f4 == _encode
use_skips=True: salida OK | parámetros del decoder = 633,716
use_skips=False: salida OK | parámetros del decoder = 608,628
use_skips=True: cambio de la salida al perturbar f1 = 1.777e+00
use_skips=False: cambio de la salida al perturbar f1 = 0.000e+00
Skip connections verificadas.


In [7]:
import copy
from src.deteccion_grid import GridDetectionHead, build_targets, grid_detection_loss, postprocess
from src.cbam_modulos import CBAM, CompuertaResidualCBAM

def joint_forward(backbone, cbam, det_head, seg_decoder, image, gt_detections=None,
                  nms_fn=None, score_thresh=0.05, iou_thresh=0.5, output_size=(256, 256)):
    """Un solo paso del backbone que alimenta las dos tareas.
    backbone -> pirámide -> (CBAM/compuerta sobre f4) -> cabeza de detección (grid) -> cajas
    -> priors de región -> decoder de segmentación con skips.
    Entrenamiento: pasa `gt_detections` (lista de dicts con boxes/labels reales).
    Inferencia: sin `gt_detections`; las cajas salen de postprocess(..., nms_fn) y NO se usa ninguna anotación."""
    features = backbone_feature_pyramid(backbone, image)
    if cbam is not None:
        features[-1] = cbam(features[-1])[0]                 # contrato: la compuerta devuelve (y, ch_attn, sp_attn)
    det_out = det_head(features[-1])                         # [B, 8, 16, 16]
    if gt_detections is not None:
        detections, source = gt_detections, "gt"
    else:
        detections = postprocess(det_out.detach(), score_thresh=score_thresh, nms_fn=nms_fn, iou_thresh=iou_thresh)
        source = "pred"
    priors = build_detection_region_priors(detections, output_size=output_size, device=image.device)
    seg_out = seg_decoder(image, features, priors, output_size=output_size)
    return {"det_out": det_out, "seg_out": seg_out, "priors": priors,
            "detections": detections, "prior_source": source}


def stack_segmentation_targets(instance_masks, device):
    """Lista de máscaras de instancia 2D -> tensores del batch para center_offset_segmentation_loss."""
    items = [build_instance_segmentation_targets(m) for m in instance_masks]
    stack = lambda key, dtype=None: torch.from_numpy(np.stack([it[key] for it in items])).to(device)
    return {"semantic": stack("semantic"), "centers": stack("centers"),
            "offsets": stack("offsets"), "foreground": stack("foreground")}


def joint_loss(joint, det_targets, seg_targets, det_lambdas=None, seg_weight=1.0):
    """total = pérdida de detección (grid_detection_loss) + seg_weight * pérdida de segmentación.
    seg_weight=1.0 es un punto de partida, NO un valor calibrado: hay que justificarlo con `val`."""
    det = grid_detection_loss(joint["det_out"], det_targets) if det_lambdas is None \
        else grid_detection_loss(joint["det_out"], det_targets, lambdas=det_lambdas)
    seg = center_offset_segmentation_loss(
        joint["seg_out"], seg_targets["semantic"], seg_targets["centers"],
        seg_targets["offsets"], seg_targets["foreground"])
    return {"total": det["total"] + seg_weight * seg["total"], "det": det, "seg": seg}

## 4. Pérdidas para las tres señales de segmentación

La pérdida semántica inicia con entropía cruzada multiclase por píxel. Los centros usan una pérdida focal tipo CenterNet, equilibrada entre positivos y negativos, para que el fondo no domine los pocos píxeles centrales. Los offsets usan Smooth L1 únicamente en píxeles óseos. Los pesos comienzan iguales como punto de partida, no como valores óptimos: se deben revisar las escalas y curvas con `val`. Las guías sugieren probar Focal o una combinación CE + Dice/IoU si el desbalance o los falsos positivos lo justifican; cualquier comparación debe usar la misma partición de validación, sin escoger por resultados de test.

**Implementado:** `center_offset_segmentation_loss(..., semantic_loss="ce_dice")` suma a la entropía cruzada un Dice suave sobre las tres regiones óseas (calculado sobre todo el lote por clase, con suavizado, excluyendo el fondo). El valor por defecto sigue siendo `"ce"` para no cambiar los resultados anteriores. Cuál de las dos variantes se queda se decide comparando ambas en `val`; todavía no se ha hecho esa comparación sobre el modelo entrenado.

In [8]:
def soft_dice_semantic_loss(semantic_logits, semantic_target, eps=1.0):
    """Dice suave (1 - Dice medio) sobre las tres regiones óseas (sacro, coxal izq., coxal der.).
    Se calcula sobre todo el lote por clase (no por corte) y con suavizado eps, para que un corte con una región
    pequeña no domine; el fondo se excluye porque ya está bien representado en la entropía cruzada."""
    probabilities = torch.softmax(semantic_logits.float(), dim=1)
    one_hot = F.one_hot(semantic_target.long(), num_classes=probabilities.shape[1]).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = (probabilities * one_hot).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + one_hot.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice[1:].mean()


def center_offset_segmentation_loss(outputs, semantic_target, center_target,
                                     offset_target, foreground_target,
                                     weights=(1.0, 1.0, 1.0),
                                     semantic_loss="ce", dice_weight=1.0):
    """semantic_loss: "ce" (entropía cruzada, comportamiento original) | "ce_dice" (CE + dice_weight * Dice suave)."""
    if semantic_loss not in ("ce", "ce_dice"):
        raise ValueError(f"semantic_loss debe ser 'ce' o 'ce_dice', se recibió {semantic_loss!r}")
    semantic_target = semantic_target.long()
    center_target = center_target.float()
    offset_target = offset_target.float()
    foreground_target = foreground_target.bool()

    semantic_ce = F.cross_entropy(outputs["semantic_logits"].float(), semantic_target)
    semantic_loss_value = semantic_ce
    extra = {}
    if semantic_loss == "ce_dice":
        semantic_dice = soft_dice_semantic_loss(outputs["semantic_logits"], semantic_target)
        semantic_loss_value = semantic_ce + dice_weight * semantic_dice
        extra = {"semantic_ce": semantic_ce, "semantic_dice": semantic_dice}

    center_probability = torch.sigmoid(outputs["center_logits"].float()).clamp(1e-5, 1 - 1e-5)
    positive = center_target.eq(1.0).float()
    negative = center_target.lt(1.0).float()
    negative_weight = (1.0 - center_target).pow(4)
    positive_loss = torch.log(center_probability) * (1.0 - center_probability).pow(2) * positive
    negative_loss = torch.log(1.0 - center_probability) * center_probability.pow(2) * negative_weight * negative
    n_positive = positive.sum().clamp_min(1.0)
    n_negative = negative.sum().clamp_min(1.0)
    positive_term = -positive_loss.sum() / n_positive
    negative_term = -negative_loss.sum() / n_negative
    center_loss = positive_term + negative_term

    offset_prediction = outputs["offsets"].float().permute(0, 2, 3, 1)
    offset_truth = offset_target.permute(0, 2, 3, 1)
    if foreground_target.any():
        offset_loss = F.smooth_l1_loss(
            offset_prediction[foreground_target], offset_truth[foreground_target]
        )
    else:
        offset_loss = offset_prediction.sum() * 0.0

    total = weights[0] * semantic_loss_value + weights[1] * center_loss + weights[2] * offset_loss
    return {"total": total, "semantic": semantic_loss_value, "center": center_loss, "offset": offset_loss, **extra}

In [9]:
from scipy.ndimage import maximum_filter


def decode_center_offset_instances(outputs, center_threshold=0.25,
                                   min_center_distance=7, min_instance_pixels=1,
                                   fallback_components=True, return_scores=False, batch_index=0):
    """Salidas del decoder -> máscara de IDs por corte.
    fallback_components: si una región semántica no tiene ningún pico de centro >= center_threshold, en vez de
        descartarla entera se separa por componentes conectados (cada uno con la confianza de su mayor valor de centro).
    return_scores: añade un cuarto resultado {id: confianza} (valor del pico de centro) para calcular AP de instancias.
    batch_index: elemento del lote a decodificar (antes siempre era el 0)."""
    from scipy.ndimage import label as _label_components
    b = int(batch_index)
    semantic_probability = torch.softmax(outputs["semantic_logits"].float(), dim=1)[b].detach().cpu().numpy()
    center_probability = torch.sigmoid(outputs["center_logits"].float())[b].detach().cpu().numpy()
    offsets = outputs["offsets"][b].detach().cpu().numpy()
    semantic_prediction = semantic_probability.argmax(axis=0).astype(np.uint8)
    height, width = semantic_prediction.shape
    instance_prediction = np.zeros((height, width), dtype=np.uint16)
    instance_classes, instance_scores = {}, {}
    next_instance_id = 1

    for class_id in (1, 2, 3):
        region_mask = semantic_prediction == class_id
        if not region_mask.any():
            continue
        heatmap = center_probability[class_id - 1]
        local_maxima = heatmap == maximum_filter(heatmap, size=2 * min_center_distance + 1)
        peak_yx = np.argwhere(local_maxima & (heatmap >= center_threshold))
        if not len(peak_yx):
            if fallback_components:
                components, n_components = _label_components(region_mask)
                for component in range(1, n_components + 1):
                    selected = components == component
                    if int(selected.sum()) < min_instance_pixels:
                        continue
                    instance_prediction[selected] = next_instance_id
                    instance_classes[next_instance_id] = class_id
                    instance_scores[next_instance_id] = float(heatmap[selected].max())
                    next_instance_id += 1
            continue

        ys, xs = np.nonzero(region_mask)
        voted_x = np.clip(xs + offsets[0, ys, xs] * width, 0, width - 1)
        voted_y = np.clip(ys + offsets[1, ys, xs] * height, 0, height - 1)
        squared_distances = ((voted_x[:, None] - peak_yx[None, :, 1]) ** 2 +
                             (voted_y[:, None] - peak_yx[None, :, 0]) ** 2)
        assignments = squared_distances.argmin(axis=1)

        for peak_index in range(len(peak_yx)):
            selected = assignments == peak_index
            if int(selected.sum()) < min_instance_pixels:
                continue
            instance_prediction[ys[selected], xs[selected]] = next_instance_id
            instance_classes[next_instance_id] = class_id
            instance_scores[next_instance_id] = float(heatmap[peak_yx[peak_index, 0], peak_yx[peak_index, 1]])
            next_instance_id += 1

    if return_scores:
        return instance_prediction, semantic_prediction, instance_classes, instance_scores
    return instance_prediction, semantic_prediction, instance_classes

## 5. Revisar formas y objetivos con un corte real de entrenamiento

Esta celda no entrena ni presenta una predicción como resultado. Carga una muestra real del split `train`, crea sus mapas objetivo y comprueba que el decoder acepte las características del backbone y los priors construidos con cajas anotadas (condicionamiento de entrenamiento), y devuelva mapas de 256 × 256. Las salidas del decoder están sin entrenar; solo se validan estructura y dimensiones.

In [10]:
segmentation_train_dataset = PengwinDetectionDataset(
    split="train", image_size=256, channels=1, as_torch=False
)
real_image, real_detection_target = segmentation_train_dataset[0]
real_segmentation_target = build_instance_segmentation_targets(
    real_detection_target["instance_mask"]
)

segmentation_decoder = CenterOffsetSkipDecoder(in_channels=256, image_channels=1).to(DEVICE)
segmentation_decoder.eval()
real_region_priors = build_detection_region_priors(
    [real_detection_target], output_size=(256, 256), device=DEVICE
)
MODELS["base"].eval()
real_image_tensor = torch.from_numpy(real_image[None, ...]).to(DEVICE)
with torch.no_grad():
    real_pyramid = backbone_feature_pyramid(MODELS["base"], real_image_tensor)
    real_feature_map = real_pyramid[-1]
    real_segmentation_output = run_skip_segmentation(
        MODELS["base"], segmentation_decoder, real_image_tensor, real_region_priors
    )

assert real_feature_map.shape == (1, 256, 16, 16), real_feature_map.shape
assert real_segmentation_output["semantic_logits"].shape == (1, 4, 256, 256)
assert real_segmentation_output["center_logits"].shape == (1, 3, 256, 256)
assert real_segmentation_output["offsets"].shape == (1, 2, 256, 256)
assert real_segmentation_target["semantic"].shape == (256, 256)
print("Caso/corte real:", real_detection_target["case_id"], real_detection_target["slice_index"])
print("IDs de fragmento en la máscara real:", real_segmentation_target["instance_ids"].tolist())
print("Pirámide:", [tuple(f.shape[1:]) for f in real_pyramid])
print("Salidas del decoder:", {k: tuple(v.shape) for k, v in real_segmentation_output.items()})
print("Priors de región:", tuple(real_region_priors.shape), "| píxeles por clase:", real_region_priors.sum(dim=(0, 2, 3)).tolist())
print("Verificación estructural OK; decoder con skips todavía no entrenado.")

<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


Caso/corte real: 001 71
IDs de fragmento en la máscara real: [11]
Pirámide: [(32, 128, 128), (64, 64, 64), (128, 32, 32), (256, 16, 16)]
Salidas del decoder: {'semantic_logits': (1, 4, 256, 256), 'center_logits': (1, 3, 256, 256), 'offsets': (1, 2, 256, 256)}
Priors de región: (1, 3, 256, 256) | píxeles por clase: [0.0, 8.0, 0.0]
Verificación estructural OK; decoder con skips todavía no entrenado.


In [11]:
torch.manual_seed(SEED); joint_gate = CompuertaResidualCBAM(CBAM(256)).to(DEVICE)
torch.manual_seed(SEED); joint_det_head = GridDetectionHead().to(DEVICE)
torch.manual_seed(SEED); joint_decoder = CenterOffsetSkipDecoder(in_channels=256, image_channels=1).to(DEVICE)
joint_backbone = copy.deepcopy(MODELS["base"]).eval()          # copia: no se tocan las estadísticas BN del original
for _m in (joint_gate, joint_det_head, joint_decoder):
    _m.eval()
_gt = [real_detection_target]

with torch.no_grad():
    _j = joint_forward(joint_backbone, joint_gate, joint_det_head, joint_decoder, real_image_tensor, gt_detections=_gt)
    _, _fm = joint_backbone(real_image_tensor); _fm, _, _ = joint_gate(_fm)
    _ref = joint_det_head(_fm)                                    # ruta del contrato: backbone -> CBAM -> cabeza
    _j0 = joint_forward(joint_backbone, None, joint_det_head, joint_decoder, real_image_tensor, gt_detections=_gt)
    _jp = joint_forward(joint_backbone, joint_gate, joint_det_head, joint_decoder, real_image_tensor, nms_fn=None, score_thresh=0.05)
assert torch.allclose(_j["det_out"], _ref, atol=1e-6)
print("1) detección conjunta == backbone -> CBAM -> cabeza (contrato de deteccion_grid.py): OK")
assert torch.allclose(_j["seg_out"]["semantic_logits"], _j0["seg_out"]["semantic_logits"], atol=1e-6)
print("2) gamma = 0: la segmentación es idéntica a la de no usar CBAM: OK")
assert _jp["prior_source"] == "pred" and _jp["seg_out"]["semantic_logits"].shape == (1, 4, 256, 256)
print(f"3) inferencia sin anotaciones: {len(_jp['detections'][0]['boxes'])} candidatas -> priors {tuple(_jp['priors'].shape)}: OK")

# Gradientes de la pérdida conjunta hasta todas las piezas (sin actualizar pesos)
_bb_train = copy.deepcopy(MODELS["base"]).train()
for _m in (joint_gate, joint_det_head, joint_decoder):
    _m.train(); _m.zero_grad(set_to_none=True)
_det_targets = build_targets(_gt)
_seg_targets = stack_segmentation_targets([real_detection_target["instance_mask"]], DEVICE)
_jt = joint_forward(_bb_train, joint_gate, joint_det_head, joint_decoder, real_image_tensor, gt_detections=_gt)
_L = joint_loss(_jt, _det_targets, _seg_targets)
_L["total"].backward()
_g = lambda p: p.grad.norm().item() if p.grad is not None else 0.0
_norms = {"conv1 (backbone)": _g(_bb_train.conv1.weight), "gamma (compuerta)": _g(joint_gate.gamma),
          "cabeza de detección": _g(joint_det_head.pred.weight), "stem (decoder)": _g(joint_decoder.stem[0].weight)}
assert all(np.isfinite(v) and v > 0 for v in _norms.values()), _norms
print("4) pérdida conjunta:", {"total": round(_L["total"].item(), 3), "det": round(_L["det"]["total"].item(), 3),
                              "seg": round(_L["seg"]["total"].item(), 3)})
print("   normas de gradiente:", {k: f"{v:.2e}" for k, v in _norms.items()})
print("Integración detección + CBAM + segmentación con skips verificada (no se actualizó ningún peso).")

1) detección conjunta == backbone -> CBAM -> cabeza (contrato de deteccion_grid.py): OK
2) gamma = 0: la segmentación es idéntica a la de no usar CBAM: OK
3) inferencia sin anotaciones: 0 candidatas -> priors (1, 3, 256, 256): OK
4) pérdida conjunta: {'total': 12.084, 'det': 7.899, 'seg': 4.185}
   normas de gradiente: {'conv1 (backbone)': '6.74e+00', 'gamma (compuerta)': '4.85e-02', 'cabeza de detección': '1.43e+01', 'stem (decoder)': '5.40e+00'}
Integración detección + CBAM + segmentación con skips verificada (no se actualizó ningún peso).


## 6. Prueba oráculo con fragmentos reales de train y val

Esta prueba usa cortes reales seleccionados por número de instancias en `train` y `val`. Convierte sus máscaras expertas en señales ideales de semántica, centros y offsets y comprueba si el postprocesamiento recupera un ID distinto para cada fragmento, incluidos los pequeños. Es una prueba de consistencia del algoritmo con anotaciones conocidas, **no** una predicción del modelo ni una métrica de desempeño. El conjunto `test` no se consulta.

Las métricas de la sección 7 se verifican contra este mismo oráculo (deben dar 1.0).

In [12]:
def check_instance_decoder_against_real_masks(split_name):
    candidates = slice_manifest[slice_manifest["split"] == split_name]
    selected = candidates.sort_values(
        ["n_instances", "n_regions"], ascending=False
    ).iloc[0]
    dataset = PengwinDetectionDataset(
        split=split_name, image_size=256, channels=1, as_torch=False
    )
    sample_index = next(
        i for i, record in enumerate(dataset.records)
        if record["case_id"] == selected["case_id"]
        and int(record["slice_index"]) == int(selected["slice_index"])
    )
    _, detection_target = dataset[sample_index]
    target = build_instance_segmentation_targets(detection_target["instance_mask"])
    height, width = target["semantic"].shape

    semantic_logits = np.full((1, 4, height, width), -20.0, dtype=np.float32)
    for class_id in range(4):
        semantic_logits[0, class_id][target["semantic"] == class_id] = 20.0
    centers = np.clip(target["centers"], 1e-4, 1 - 1e-4)
    center_logits = np.log(centers / (1 - centers))[None]
    oracle_outputs = {
        "semantic_logits": torch.from_numpy(semantic_logits),
        "center_logits": torch.from_numpy(center_logits),
        "offsets": torch.from_numpy(target["offsets"][None]),
    }
    predicted_instances, predicted_semantic, _ = decode_center_offset_instances(
        oracle_outputs, min_instance_pixels=1
    )

    assert np.array_equal(predicted_semantic, target["semantic"])
    true_ids = target["instance_ids"]
    for instance_id in true_ids:
        pixels = detection_target["instance_mask"] == instance_id
        assigned_ids = np.unique(predicted_instances[pixels])
        assert len(assigned_ids) == 1 and assigned_ids[0] != 0, (
            split_name, int(instance_id), assigned_ids.tolist()
        )
    predicted_ids = np.unique(predicted_instances)
    predicted_ids = predicted_ids[predicted_ids != 0]
    assert len(predicted_ids) == len(true_ids), (split_name, true_ids, predicted_ids)

    areas = [int((detection_target["instance_mask"] == iid).sum()) for iid in true_ids]
    print(
        f"{split_name}: caso {selected['case_id']} / corte {selected['slice_index']} | "
        f"{len(true_ids)} fragmentos reconstruidos; área mínima {min(areas)} px"
    )


for split_name in ("train", "val"):
    check_instance_decoder_against_real_masks(split_name)
print("Prueba oráculo OK. No equivale a inferencia ni a desempeño del modelo.")


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


train: caso 014 / corte 204 | 7 fragmentos reconstruidos; área mínima 7 px
val: caso 079 / corte 63 | 6 fragmentos reconstruidos; área mínima 8 px
Prueba oráculo OK. No equivale a inferencia ni a desempeño del modelo.


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


## 7. Métricas de segmentación

Las métricas se definen en la celda siguiente (numpy puro). `entrenamiento_multitarea.ipynb` las carga de aquí por nombre, igual que el resto de definiciones del equipo, así que se calcula exactamente lo mismo en ambos cuadernos.

- **Semántica:** IoU y Dice por región (sacro, coxal izquierdo, coxal derecho), acumulando intersección y unión sobre todos los cortes, no promediando por corte.
- **Instancias (2D por corte):** AP de máscara a IoU 0.5 y promedio 0.5:0.95; PQ/SQ/RQ (calidad panóptica: SQ = IoU medio de los fragmentos emparejados, RQ = F1 de emparejamiento); recall de fragmentos por tamaño (pequeño < 25 px, mediano, grande) porque los fragmentos pequeños son los difíciles. El emparejamiento exige la **misma región** anatómica. La confianza de cada instancia es el valor de su pico en el mapa de centros.
- **Taxonomía de errores**, por fragmento real: `ok` (IoU ≥ 0.5 y región correcta), `clase_incorrecta`, `fusionado` (una instancia predicha cubre ≥ 50 % de dos o más fragmentos reales), `dividido` (≥ 2 instancias predichas cubren ≥ 20 % del fragmento y ninguna llega a IoU 0.5), `omitido` (< 20 % cubierto), `mal_segmentado` (resto) y `falso_positivo` por instancia predicha (< 10 % de su área sobre hueso real). Los umbrales 50 % / 20 % / 10 % son convenciones de este proyecto, no estándares.

**Verificación:** (1) con la señal oráculo de un corte real, todas las métricas deben dar 1.0; (2) con errores provocados a propósito sobre una máscara sintética (omitir, fusionar, dividir, añadir un falso positivo, cambiar la región) cada contador debe reaccionar como corresponde. Esto comprueba el código de las métricas, no la calidad del modelo.

`decode_center_offset_instances` ahora acepta `fallback_components` (si una región no tiene ningún pico de centro, se separa por componentes conectados en vez de descartarse) y `return_scores`.

In [13]:
# Métricas de segmentación (numpy). Convención de IDs de instancia: 1..10 sacro, 11..20 coxal izquierdo, 21..30 coxal derecho.
# Todas las métricas de instancia son 2D por corte (256 x 256); no son métricas volumétricas.
SEG_CLASSES = (1, 2, 3)
SEG_CLASS_NAMES = {1: "sacro", 2: "coxal_izquierdo", 3: "coxal_derecho"}
INSTANCE_IOU_THRESHOLDS = tuple(np.round(np.arange(0.5, 0.96, 0.05), 2))
FRAGMENT_SIZE_BINS = ((0, 25), (25, 100), (100, 10 ** 9))          # área del fragmento GT en píxeles (a 256 x 256)
FRAGMENT_SIZE_BIN_NAMES = ("pequeño(<25px)", "mediano(25-100px)", "grande(>=100px)")


def class_of_instance_id(instance_id):
    instance_id = int(instance_id)
    if 1 <= instance_id <= 10:
        return 1
    if 11 <= instance_id <= 20:
        return 2
    if 21 <= instance_id <= 30:
        return 3
    raise ValueError(f"ID de fragmento fuera de 1..30: {instance_id}")


# --------------------------------------------------------------------------- semántica
class SemanticAccumulator:
    """IoU y Dice por región acumulando intersección/unión sobre todos los cortes (no promedia por corte)."""

    def __init__(self, classes=SEG_CLASSES):
        self.classes = tuple(classes)
        self.counts = {c: {"inter": 0.0, "psum": 0.0, "gsum": 0.0} for c in self.classes}

    def add(self, pred, gt):
        pred, gt = np.asarray(pred), np.asarray(gt)
        if pred.shape != gt.shape:
            raise ValueError(f"formas distintas: {pred.shape} vs {gt.shape}")
        for c in self.classes:
            p, g = pred == c, gt == c
            self.add_counts(c, float((p & g).sum()), float(p.sum()), float(g.sum()))

    def add_counts(self, c, inter, psum, gsum):
        a = self.counts[c]
        a["inter"] += inter; a["psum"] += psum; a["gsum"] += gsum

    def summary(self):
        out, dices, ious = {}, [], []
        for c in self.classes:
            a = self.counts[c]
            denom, union = a["psum"] + a["gsum"], a["psum"] + a["gsum"] - a["inter"]
            dice = 2 * a["inter"] / denom if denom > 0 else float("nan")
            iou = a["inter"] / union if union > 0 else float("nan")
            out[f"dice_c{c}"], out[f"iou_c{c}"] = dice, iou
            dices.append(dice); ious.append(iou)
        out["dice_mean"] = float(np.nanmean(dices)) if not np.all(np.isnan(dices)) else float("nan")
        out["iou_mean"] = float(np.nanmean(ious)) if not np.all(np.isnan(ious)) else float("nan")
        return out


# --------------------------------------------------------------------------- instancias
def pairwise_overlap(gt_inst, pred_inst):
    """(gt_ids, pred_ids, inter[G,P], gt_area[G], pred_area[P]) entre dos máscaras de IDs (0 = fondo)."""
    gt_inst = np.asarray(gt_inst).astype(np.int64)
    pred_inst = np.asarray(pred_inst).astype(np.int64)
    if gt_inst.shape != pred_inst.shape:
        raise ValueError(f"formas distintas: {gt_inst.shape} vs {pred_inst.shape}")
    gt_ids = np.unique(gt_inst[gt_inst > 0])
    pred_ids = np.unique(pred_inst[pred_inst > 0])
    G, P = len(gt_ids), len(pred_ids)
    if G == 0 or P == 0:
        ga = np.array([(gt_inst == i).sum() for i in gt_ids], dtype=np.float64)
        pa = np.array([(pred_inst == i).sum() for i in pred_ids], dtype=np.float64)
        return gt_ids, pred_ids, np.zeros((G, P)), ga, pa
    lut_g = np.full(int(gt_inst.max()) + 1, -1, dtype=np.int64); lut_g[gt_ids] = np.arange(G)
    lut_p = np.full(int(pred_inst.max()) + 1, -1, dtype=np.int64); lut_p[pred_ids] = np.arange(P)
    gi, pi = lut_g[gt_inst], lut_p[pred_inst]
    ga = np.bincount(gi[gi >= 0], minlength=G).astype(np.float64)
    pa = np.bincount(pi[pi >= 0], minlength=P).astype(np.float64)
    both = (gi >= 0) & (pi >= 0)
    inter = np.bincount(gi[both] * P + pi[both], minlength=G * P).reshape(G, P).astype(np.float64)
    return gt_ids, pred_ids, inter, ga, pa


def _ap_from_pr_curve(recall, precision):
    mrec = np.concatenate([[0.0], recall, [1.0]]); mpre = np.concatenate([[0.0], precision, [0.0]])
    for k in range(len(mpre) - 2, -1, -1):
        mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


class InstanceEvaluator:
    """Acumula cortes y calcula: AP de máscara (IoU 0.5 y 0.5:0.95), PQ/SQ/RQ, recall de fragmentos por tamaño
    y una taxonomía de errores (omitido, fusionado, dividido, mal segmentado, clase incorrecta, falso positivo).

    add(pred_inst, gt_inst, pred_classes, pred_scores=None)
      pred_inst:    máscara 2D de IDs predichos (0 = fondo)
      gt_inst:      máscara 2D con los IDs reales 1..30
      pred_classes: dict id_predicho -> región 1..3
      pred_scores:  dict id_predicho -> confianza (sin ella, todas valen 1 y el AP pierde significado de ranking)
    """

    def __init__(self, classes=SEG_CLASSES, merge_cover=0.5, split_cover=0.2, omit_cover=0.2, fp_cover=0.1):
        self.classes = tuple(classes)
        self.merge_cover, self.split_cover, self.omit_cover, self.fp_cover = merge_cover, split_cover, omit_cover, fp_cover
        self.slices = []                      # por corte y clase: (iou[G,P] con clase, scores[P])
        self.n_slices = 0
        self.errors = {k: 0 for k in ("ok", "clase_incorrecta", "fusionado", "dividido", "omitido",
                                      "mal_segmentado", "falso_positivo")}
        self.size_stats = [{"n": 0, "ok": 0} for _ in FRAGMENT_SIZE_BINS]
        self.class_stats = {c: {"n_gt": 0, "n_pred": 0, "tp": 0, "iou_sum": 0.0} for c in self.classes}

    def add(self, pred_inst, gt_inst, pred_classes, pred_scores=None):
        pred_classes = dict(pred_classes or {})
        pred_scores = dict(pred_scores or {})
        gt_ids, pred_ids, inter, ga, pa = pairwise_overlap(gt_inst, pred_inst)
        self.n_slices += 1
        G, P = len(gt_ids), len(pred_ids)
        gt_cls = np.array([class_of_instance_id(i) for i in gt_ids], dtype=np.int64)
        pr_cls = np.array([int(pred_classes.get(int(i), 0)) for i in pred_ids], dtype=np.int64)
        pr_score = np.array([float(pred_scores.get(int(i), 1.0)) for i in pred_ids], dtype=np.float64)
        union = ga[:, None] + pa[None, :] - inter
        iou_raw = np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)
        iou_cls = iou_raw * (gt_cls[:, None] == pr_cls[None, :])             # emparejar solo dentro de la misma región

        for c in self.classes:
            gi, pj = np.flatnonzero(gt_cls == c), np.flatnonzero(pr_cls == c)
            self.slices.append((c, iou_cls[np.ix_(gi, pj)], pr_score[pj]))
            sub = iou_cls[np.ix_(gi, pj)]
            tp_mask = sub > 0.5
            st = self.class_stats[c]
            st["n_gt"] += len(gi); st["n_pred"] += len(pj)
            st["tp"] += int(tp_mask.sum()); st["iou_sum"] += float(sub[tp_mask].sum())

        # taxonomía de errores, fragmento a fragmento
        for g in range(G):
            row = inter[g]
            best_p = int(row.argmax()) if P else -1
            best_iou = float(iou_raw[g, best_p]) if P else 0.0
            if P and best_iou >= 0.5:
                kind = "ok" if pr_cls[best_p] == gt_cls[g] else "clase_incorrecta"
            else:
                cover = float(row.sum() / ga[g]) if P else 0.0
                merged = False
                if P:
                    p_star = int(row.argmax())
                    contained = (inter[:, p_star] / ga) >= self.merge_cover
                    merged = bool(contained.sum() >= 2 and inter[g, p_star] / ga[g] >= self.merge_cover)
                n_parts = int(((row / ga[g]) >= self.split_cover).sum()) if P else 0
                if merged:
                    kind = "fusionado"
                elif cover < self.omit_cover:
                    kind = "omitido"
                elif n_parts >= 2:
                    kind = "dividido"
                else:
                    kind = "mal_segmentado"
            self.errors[kind] += 1
            for b, (lo, hi) in enumerate(FRAGMENT_SIZE_BINS):
                if lo <= ga[g] < hi:
                    self.size_stats[b]["n"] += 1; self.size_stats[b]["ok"] += int(kind == "ok")
        for p in range(P):
            if inter[:, p].sum() / pa[p] < self.fp_cover:
                self.errors["falso_positivo"] += 1

    # ------------------------------------------------------------------ resumen
    def _ap(self, c, thr):
        gts_total = sum(m.shape[0] for cc, m, _ in self.slices if cc == c)
        if gts_total == 0:
            return None
        preds = [(float(s), k, j) for k, (cc, m, sc) in enumerate(self.slices) if cc == c for j, s in enumerate(sc)]
        preds.sort(key=lambda d: -d[0])
        used = {k: np.zeros(m.shape[0], dtype=bool) for k, (cc, m, _) in enumerate(self.slices) if cc == c}
        tp, fp = np.zeros(len(preds)), np.zeros(len(preds))
        for n, (_, k, j) in enumerate(preds):
            col = np.where(used[k], -1.0, self.slices[k][1][:, j]) if self.slices[k][1].shape[0] else np.array([])
            if len(col) and col.max() >= thr:
                used[k][int(col.argmax())] = True; tp[n] = 1
            else:
                fp[n] = 1
        if not len(preds):
            return 0.0
        tp, fp = np.cumsum(tp), np.cumsum(fp)
        return _ap_from_pr_curve(tp / gts_total, tp / np.maximum(tp + fp, 1e-9))

    def summary(self):
        out = {"n_slices": self.n_slices}
        ap50, ap_all = {}, {}
        for c in self.classes:
            ap50[c] = self._ap(c, 0.5)
            vals = [self._ap(c, t) for t in INSTANCE_IOU_THRESHOLDS]
            ap_all[c] = None if vals[0] is None else float(np.mean(vals))
            out[f"ap50_c{c}"] = float("nan") if ap50[c] is None else ap50[c]
            out[f"ap50_95_c{c}"] = float("nan") if ap_all[c] is None else ap_all[c]
        v50 = [v for v in ap50.values() if v is not None]; vall = [v for v in ap_all.values() if v is not None]
        out["ap50"] = float(np.mean(v50)) if v50 else float("nan")
        out["ap50_95"] = float(np.mean(vall)) if vall else float("nan")

        pqs, sqs, rqs = [], [], []
        tp_all = fp_all = fn_all = 0
        iou_sum_all = 0.0
        for c in self.classes:
            st = self.class_stats[c]
            tp, fp, fn = st["tp"], st["n_pred"] - st["tp"], st["n_gt"] - st["tp"]
            tp_all += tp; fp_all += fp; fn_all += fn; iou_sum_all += st["iou_sum"]
            if st["n_gt"] == 0 and st["n_pred"] == 0:
                sq = rq = pq = float("nan")
            else:
                sq = st["iou_sum"] / tp if tp else 0.0
                rq = tp / (tp + 0.5 * fp + 0.5 * fn)
                pq = sq * rq
            out[f"pq_c{c}"], out[f"sq_c{c}"], out[f"rq_c{c}"] = pq, sq, rq
            out[f"n_gt_c{c}"], out[f"n_pred_c{c}"] = st["n_gt"], st["n_pred"]
            pqs.append(pq); sqs.append(sq); rqs.append(rq)
        out["pq"], out["sq"], out["rq"] = (float(np.nanmean(v)) if not np.all(np.isnan(v)) else float("nan")
                                           for v in (pqs, sqs, rqs))
        n_gt = sum(self.class_stats[c]["n_gt"] for c in self.classes)
        out["n_gt"] = n_gt
        out["recall50"] = self.errors["ok"] / n_gt if n_gt else float("nan")
        for name, st in zip(FRAGMENT_SIZE_BIN_NAMES, self.size_stats):
            out[f"recall50_{name}"] = st["ok"] / st["n"] if st["n"] else float("nan")
            out[f"n_gt_{name}"] = st["n"]
        for k, v in self.errors.items():
            out[f"err_{k}"] = v
        return out


def format_instance_summary(s, title=""):
    """Texto corto para imprimir un resumen de InstanceEvaluator.summary()."""
    lines = [title] if title else []
    lines.append(f"  AP50={s['ap50']:.3f}  AP50:95={s['ap50_95']:.3f}  PQ={s['pq']:.3f} (SQ={s['sq']:.3f}, RQ={s['rq']:.3f})  "
                 f"recall@0.5={s['recall50']:.3f}  [{s['n_gt']} fragmentos GT, {s['n_slices']} cortes]")
    lines.append("  recall por tamaño: " + ", ".join(f"{n}={s[f'recall50_{n}']:.2f} (n={s[f'n_gt_{n}']})" for n in FRAGMENT_SIZE_BIN_NAMES))
    lines.append("  errores: " + ", ".join(f"{k[4:]}={s[k]}" for k in s if k.startswith("err_")))
    return "\n".join(lines)

In [14]:
# ---- 1) Errores provocados a propósito sobre una máscara sintética (cada contador debe reaccionar) ----
def _synthetic_gt():
    gt = np.zeros((64, 64), np.uint16)
    gt[5:25, 5:25] = 1; gt[5:25, 30:50] = 2; gt[40:50, 5:20] = 11; gt[40:44, 40:44] = 21
    return gt

_gt = _synthetic_gt()
_cls = {1: 1, 2: 1, 11: 2, 21: 3}

def _run(pred, classes, scores=None):
    evaluator = InstanceEvaluator(); evaluator.add(pred, _gt, classes, scores); return evaluator.summary()

_s = _run(_gt.copy(), _cls)
assert _s["ap50"] == 1 and _s["ap50_95"] == 1 and _s["pq"] == 1 and _s["err_ok"] == 4 and _s["recall50"] == 1
_p = _gt.copy(); _p[_p == 2] = 0
_s = _run(_p, _cls); assert _s["err_omitido"] == 1 and _s["err_ok"] == 3 and _s["recall50"] == 0.75
_p = _gt.copy(); _p[_p == 2] = 1; _p[5:25, 25:30] = 1
_s = _run(_p, {1: 1, 11: 2, 21: 3}); assert _s["err_fusionado"] >= 1
_p = _gt.copy(); _p[40:50, 10:15] = 12; _p[40:50, 15:20] = 13
_s = _run(_p, {**_cls, 12: 2, 13: 2}); assert _s["err_dividido"] == 1
_p = _gt.copy(); _p[58:62, 58:62] = 30
_s = _run(_p, {**_cls, 30: 3}); assert _s["err_falso_positivo"] == 1 and _s["err_ok"] == 4 and _s["rq"] < 1
_s = _run(_gt.copy(), {**_cls, 11: 3}); assert _s["err_clase_incorrecta"] == 1 and _s["err_ok"] == 3
_alto = _run(_p, {**_cls, 30: 3}, {1: .9, 2: .9, 11: .9, 21: .9, 30: .99})["ap50"]
_bajo = _run(_p, {**_cls, 30: 3}, {1: .9, 2: .9, 11: .9, 21: .9, 30: .01})["ap50"]
assert _bajo == 1.0 and _alto < _bajo, "un falso positivo con confianza alta debe bajar el AP más que uno con confianza baja"
_p = _gt.copy(); _p[5:25, 5:11] = 0
_s = _run(_p, _cls); assert _s["ap50"] == 1 and _s["ap50_95"] < 1
_acc = SemanticAccumulator(); _sem = np.zeros((8, 8), int); _sem[:4] = 1; _pred_sem = _sem.copy(); _pred_sem[:2] = 0
_acc.add(_pred_sem, _sem); _r = _acc.summary()
assert abs(_r["dice_c1"] - 2 / 3) < 1e-9 and abs(_r["iou_c1"] - 0.5) < 1e-9 and np.isnan(_r["dice_c2"])
print("Métricas: errores provocados -> contadores correctos: OK")


# ---- 2) Pérdida CE + Dice: valores y gradiente ----
_logits = torch.randn(2, 4, 16, 16, requires_grad=True)
_target = torch.randint(0, 4, (2, 16, 16))
_perfect = F.one_hot(_target, 4).permute(0, 3, 1, 2).float() * 40.0
assert soft_dice_semantic_loss(_perfect, _target).item() < 1e-3
_dice = soft_dice_semantic_loss(_logits, _target); _dice.backward()
assert torch.isfinite(_logits.grad).all() and _logits.grad.abs().sum() > 0
print(f"Dice suave: perfecto ≈ 0 (OK) | logits aleatorios = {_dice.item():.3f} (gradiente finito)")


# ---- 3) Oráculo: señal ideal -> decodificador -> métricas deben dar 1.0 ----
def oracle_outputs_from_instance_mask(instance_mask):
    target = build_instance_segmentation_targets(instance_mask)
    height, width = target["semantic"].shape
    semantic_logits = np.full((1, 4, height, width), -20.0, dtype=np.float32)
    for class_id in range(4):
        semantic_logits[0, class_id][target["semantic"] == class_id] = 20.0
    centers = np.clip(target["centers"], 1e-4, 1 - 1e-4)
    return {"semantic_logits": torch.from_numpy(semantic_logits),
            "center_logits": torch.from_numpy(np.log(centers / (1 - centers))[None]),
            "offsets": torch.from_numpy(target["offsets"][None])}, target


def oracle_metrics_for_instance_mask(instance_mask):
    outputs, target = oracle_outputs_from_instance_mask(instance_mask)
    pred_inst, pred_sem, pred_cls, pred_scores = decode_center_offset_instances(outputs, return_scores=True)
    evaluator = InstanceEvaluator(); evaluator.add(pred_inst, instance_mask, pred_cls, pred_scores)
    semantic = SemanticAccumulator(); semantic.add(pred_sem, target["semantic"])
    return evaluator.summary(), semantic.summary()


_inst, _sem_s = oracle_metrics_for_instance_mask(_gt.astype(np.int32))      # prueba sintética (siempre disponible)
assert _inst["ap50"] == 1 and _inst["pq"] == 1 and _inst["recall50"] == 1 and _sem_s["dice_mean"] == 1
print("Oráculo sintético: AP50 = PQ = recall = Dice = 1.0: OK")

for split_name in ("train", "val"):                                           # mismos cortes reales que la prueba de la sección 6
    candidates = slice_manifest[slice_manifest["split"] == split_name]
    selected = candidates.sort_values(["n_instances", "n_regions"], ascending=False).iloc[0]
    dataset = PengwinDetectionDataset(split=split_name, image_size=256, channels=1, as_torch=False)
    sample_index = next(i for i, record in enumerate(dataset.records)
                        if record["case_id"] == selected["case_id"] and int(record["slice_index"]) == int(selected["slice_index"]))
    _, detection_target = dataset[sample_index]
    inst_summary, sem_summary = oracle_metrics_for_instance_mask(detection_target["instance_mask"])
    assert inst_summary["ap50"] == 1 and inst_summary["pq"] == 1 and inst_summary["recall50"] == 1, inst_summary
    assert sem_summary["dice_mean"] == 1 and sem_summary["iou_mean"] == 1, sem_summary
    print(format_instance_summary(inst_summary, f"Oráculo real {split_name} (caso {selected['case_id']}, corte {selected['slice_index']}):"))
print("Métricas verificadas con el oráculo. Siguen sin medir al modelo: eso se hace en entrenamiento_multitarea.ipynb.")

Métricas: errores provocados -> contadores correctos: OK
Dice suave: perfecto ≈ 0 (OK) | logits aleatorios = 0.749 (gradiente finito)
Oráculo sintético: AP50 = PQ = recall = Dice = 1.0: OK


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


Oráculo real train (caso 014, corte 204):
  AP50=1.000  AP50:95=1.000  PQ=1.000 (SQ=1.000, RQ=1.000)  recall@0.5=1.000  [7 fragmentos GT, 1 cortes]
  recall por tamaño: pequeño(<25px)=1.00 (n=1), mediano(25-100px)=1.00 (n=2), grande(>=100px)=1.00 (n=4)
  errores: ok=7, clase_incorrecta=0, fusionado=0, dividido=0, omitido=0, mal_segmentado=0, falso_positivo=0
Oráculo real val (caso 079, corte 63):
  AP50=1.000  AP50:95=1.000  PQ=1.000 (SQ=1.000, RQ=1.000)  recall@0.5=1.000  [6 fragmentos GT, 1 cortes]
  recall por tamaño: pequeño(<25px)=1.00 (n=2), mediano(25-100px)=nan (n=0), grande(>=100px)=1.00 (n=4)
  errores: ok=6, clase_incorrecta=0, fusionado=0, dividido=0, omitido=0, mal_segmentado=0, falso_positivo=0
Métricas verificadas con el oráculo. Siguen sin medir al modelo: eso se hace en entrenamiento_multitarea.ipynb.


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


## 8. Comprobar pérdida y gradientes con la anotación real

Se usa la misma muestra real para calcular las tres pérdidas y hacer una propagación hacia atrás. Esto comprueba que los objetivos tienen el formato correcto y que el gradiente llega al decoder y al backbone. Es una prueba técnica de un paso, no entrenamiento ni evidencia de calidad predictiva.

In [15]:
segmentation_decoder.train()
MODELS["base"].train()
MODELS["base"].zero_grad(set_to_none=True)
segmentation_decoder.zero_grad(set_to_none=True)

real_region_priors = build_detection_region_priors(
    [real_detection_target], output_size=(256, 256), device=DEVICE
)
real_segmentation_output = run_skip_segmentation(
    MODELS["base"], segmentation_decoder, real_image_tensor, real_region_priors
)
semantic_target_tensor = torch.from_numpy(real_segmentation_target["semantic"][None]).to(DEVICE)
center_target_tensor = torch.from_numpy(real_segmentation_target["centers"][None]).to(DEVICE)
offset_target_tensor = torch.from_numpy(real_segmentation_target["offsets"][None]).to(DEVICE)
foreground_target_tensor = torch.from_numpy(real_segmentation_target["foreground"][None]).to(DEVICE)
segmentation_losses = center_offset_segmentation_loss(
    real_segmentation_output,
    semantic_target_tensor,
    center_target_tensor,
    offset_target_tensor,
    foreground_target_tensor,
)
segmentation_losses["total"].backward()
backbone_grad_norm = MODELS["base"].conv1.weight.grad.norm().item()
stem_grad_norm = segmentation_decoder.stem[0].weight.grad.norm().item()
assert all(torch.isfinite(value) for value in segmentation_losses.values())
assert np.isfinite(backbone_grad_norm) and backbone_grad_norm > 0
assert np.isfinite(stem_grad_norm) and stem_grad_norm > 0
print("Pérdidas reales:", {name: round(value.item(), 4) for name, value in segmentation_losses.items()})
print(f"Norma de gradiente en conv1 del backbone: {backbone_grad_norm:.6f}")
print(f"Norma de gradiente en el stem del decoder: {stem_grad_norm:.6f}")
print("Gradiente real hacia backbone, stem y decoder: OK; no se actualizó ningún peso.")
MODELS["base"].zero_grad(set_to_none=True)
segmentation_decoder.zero_grad(set_to_none=True)
MODELS["base"].eval()

Pérdidas reales: {'total': 1.9752, 'semantic': 1.5031, 'center': 0.2617, 'offset': 0.2104}
Norma de gradiente en conv1 del backbone: 0.677831
Norma de gradiente en el stem del decoder: 0.842432
Gradiente real hacia backbone, stem y decoder: OK; no se actualizó ningún peso.


FundidoraPC(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool4): MaxPool2d(kernel_size=2, stride=2, padding=0, dila

## 9. Alcance y estado actual

Lo implementado aquí define objetivos, decoder condicionado por priors de región (con skip connections), pérdidas, reconstrucción 2D y una utilidad de asociación entre cortes. El entrenamiento conjunto con el backbone y las demás cabezas se hace en `Entrenamiento_multitarea.ipynb`; la asociación 3D definitiva (criterio de solapamiento, saltos entre cortes, limpieza y ajuste con `val`) está en `salida_3d.ipynb`, que reemplaza a la de la sección 11 de este cuaderno. SAM y las métricas de comparación se reportan en `sam_baseline.ipynb`. No interpretar el decoder inicializado al azar de las secciones anteriores como una segmentación válida: son pruebas de forma y gradiente.

Consejos prácticos retomados de los resúmenes de clase y la guía del avance:

- Mantener la salida alineada con la imagen de entrada (aquí, 256 × 256). Al volver a la geometría del MHA, invertir correctamente el letterbox/escala y respetar orden de ejes y spacing; no medir distancias físicas sobre la máscara redimensionada sin recuperar esa geometría.
- Si se agregan recortes, flips o cambios de brillo, aplicarlos solo a `train`. Las transformaciones espaciales deben ser idénticas para imagen y anotaciones; para máscaras categóricas usar interpolación de vecino más cercano.
- Conservar fragmentos pequeños: el filtro mínimo actual es 1 píxel. Cualquier limpieza morfológica debe comprobarse con ejemplos anotados para no borrar fragmentos reales.
- Inspeccionar casos de contacto, fragmentos omitidos, fusiones y divisiones. Las métricas agregadas no bastan para detectar todos estos errores.
- Antes de entrenamiento largo, probar formas, pérdida, gradientes e integración en un lote pequeño; guardar configuración y checkpoint y escogerlos con `val`.

Los consejos sobre conexiones de salto y pérdidas alternativas son guías para la siguiente iteración, no afirmaciones de que ya estén implementadas ni garantías de mejora.

## 10. Restaurar la máscara a la cuadrícula original del MHA

El decoder trabaja en 256 × 256 por el letterbox del dataset. Antes de construir el volumen, esta función recorta el relleno y devuelve cada máscara categórica a la forma original `(y, x)` con vecino más cercano, para no inventar IDs intermedios. Se debe pasar una máscara de instancias o semántica y la transformación guardada para ese mismo corte.

In [16]:
def restore_letterbox_label_mask(label_mask, original_shape_yx, transform, output_size=256):
    label_mask = np.asarray(label_mask)
    if label_mask.ndim != 2 or label_mask.shape != (output_size, output_size):
        raise ValueError(f"Se esperaba máscara {(output_size, output_size)}, se recibió {label_mask.shape}")
    if len(original_shape_yx) != 2:
        raise ValueError("original_shape_yx debe contener (alto, ancho)")

    original_height, original_width = map(int, original_shape_yx)
    scale = float(transform["scale"])
    if original_height <= 0 or original_width <= 0 or scale <= 0:
        raise ValueError("Dimensiones originales y escala deben ser positivas")

    resized_width = max(1, int(round(original_width * scale)))
    resized_height = max(1, int(round(original_height * scale)))
    offset_x, offset_y = int(transform["offset_x"]), int(transform["offset_y"])
    x_end, y_end = offset_x + resized_width, offset_y + resized_height
    if offset_x < 0 or offset_y < 0 or x_end > output_size or y_end > output_size:
        raise ValueError("Transformación letterbox incompatible con el tamaño de salida")

    cropped = label_mask[offset_y:y_end, offset_x:x_end]
    restored = np.asarray(
        Image.fromarray(cropped.astype(np.int32)).resize(
            (original_width, original_height), Image.Resampling.NEAREST
        ),
        dtype=np.int32,
    )
    return restored


## 11. Asociar instancias 2D entre cortes y formar el volumen 3D

Los IDs generados por el decoder son locales a cada corte; no se deben asumir persistentes entre cortes. Esta función asigna IDs globales comparando instancias de la misma clase en cortes Z consecutivos. Usa solapamiento sobre el área menor (umbral inicial `min_overlap=0.10`) y una asignación uno-a-uno óptima. Ese umbral es provisional y debe revisarse con casos de `val`. Una instancia que nace sin pareja recibe un ID nuevo; un corte vacío interrumpe la trayectoria y no se rellenan huecos automáticamente.

La función también registra candidatos a fusión o división cuando una instancia tiene solapamiento suficiente con más de una instancia vecina. Esas alertas son para inspección: el emparejamiento no puede reparar una máscara que ya salió fusionada o dividida. El volumen de entrada debe incluir todos los cortes en orden Z, incluso los vacíos, y estar ya restaurado a la cuadrícula original del MHA.

In [17]:
def link_slice_instances_to_volume(instance_masks_zyx, semantic_masks_zyx, min_overlap=0.10):
    from scipy.optimize import linear_sum_assignment

    instance_masks_zyx = np.asarray(instance_masks_zyx)
    semantic_masks_zyx = np.asarray(semantic_masks_zyx)
    if instance_masks_zyx.ndim != 3 or semantic_masks_zyx.ndim != 3:
        raise ValueError("Las máscaras deben tener forma (z, y, x)")
    if instance_masks_zyx.shape != semantic_masks_zyx.shape:
        raise ValueError("Las máscaras de instancia y semántica deben tener la misma forma")
    if not 0 <= min_overlap <= 1:
        raise ValueError("min_overlap debe estar entre 0 y 1")

    volume_ids = np.zeros(instance_masks_zyx.shape, dtype=np.uint32)
    volume_classes = {}
    slice_id_maps = []
    events = []
    next_global_id = 1

    for z_index in range(instance_masks_zyx.shape[0]):
        local_slice = instance_masks_zyx[z_index]
        semantic_slice = semantic_masks_zyx[z_index]
        local_ids = [int(v) for v in np.unique(local_slice) if v != 0]
        class_by_local_id = {}
        for local_id in local_ids:
            class_values, counts = np.unique(
                semantic_slice[local_slice == local_id], return_counts=True
            )
            valid = (class_values > 0) & (class_values <= 3)
            if not valid.any():
                raise ValueError(f"Instancia {local_id} del corte {z_index} no tiene clase anatómica")
            class_by_local_id[local_id] = int(class_values[valid][np.argmax(counts[valid])])

        current_map = {}
        previous_ids = [int(v) for v in np.unique(volume_ids[z_index - 1]) if v != 0] if z_index else []
        for class_id in (1, 2, 3):
            current_class_ids = [i for i in local_ids if class_by_local_id[i] == class_id]
            previous_class_ids = [i for i in previous_ids if volume_classes[i] == class_id]
            if not current_class_ids:
                continue

            scores = np.zeros((len(previous_class_ids), len(current_class_ids)), dtype=np.float32)
            previous_masks = [volume_ids[z_index - 1] == i for i in previous_class_ids] if z_index else []
            current_masks = [local_slice == i for i in current_class_ids]
            for row, previous_mask in enumerate(previous_masks):
                previous_area = int(previous_mask.sum())
                for col, current_mask in enumerate(current_masks):
                    intersection = int(np.logical_and(previous_mask, current_mask).sum())
                    denominator = min(previous_area, int(current_mask.sum()))
                    scores[row, col] = intersection / denominator if denominator else 0.0

            for row in range(scores.shape[0]):
                linked_columns = np.flatnonzero(scores[row] >= min_overlap)
                if len(linked_columns) > 1:
                    events.append({"type": "split_candidate", "slice_index": z_index,
                                   "previous_volume_id": previous_class_ids[row],
                                   "current_local_ids": [current_class_ids[c] for c in linked_columns]})
            for col in range(scores.shape[1]):
                linked_rows = np.flatnonzero(scores[:, col] >= min_overlap)
                if len(linked_rows) > 1:
                    events.append({"type": "merge_candidate", "slice_index": z_index,
                                   "current_local_id": current_class_ids[col],
                                   "previous_volume_ids": [previous_class_ids[r] for r in linked_rows]})

            if scores.size:
                rows, columns = linear_sum_assignment(-scores)
                for row, column in zip(rows, columns):
                    if scores[row, column] >= min_overlap:
                        current_map[current_class_ids[column]] = previous_class_ids[row]

        for local_id in local_ids:
            if local_id not in current_map:
                current_map[local_id] = next_global_id
                volume_classes[next_global_id] = class_by_local_id[local_id]
                next_global_id += 1
            else:
                volume_classes[current_map[local_id]] = class_by_local_id[local_id]
            volume_ids[z_index][local_slice == local_id] = current_map[local_id]
        slice_id_maps.append(current_map)

    return volume_ids, volume_classes, slice_id_maps, events


## 12. Prueba sintética de restauración y asociación 3D

La prueba construye dos fragmentos de clases distintas que se desplazan ligeramente entre tres cortes y cambian sus IDs locales. Comprueba que el letterbox vuelva a la forma original, que cada fragmento conserve un ID global entre cortes y que la clase no cambie. No usa imágenes médicas ni reemplaza la validación con predicciones entrenadas.

In [18]:
# Prueba de inversión de letterbox con IDs categóricos
original_labels = np.zeros((40, 80), dtype=np.int16)
original_labels[10:20, 20:30] = 12
image_for_letterbox = np.zeros_like(original_labels, dtype=np.float32)
_, letterboxed_labels, letterbox_transform = letterbox_pair(
    image_for_letterbox, original_labels, output_size=256
)
restored_labels = restore_letterbox_label_mask(
    letterboxed_labels, original_labels.shape, letterbox_transform
)
assert restored_labels.shape == original_labels.shape
assert set(np.unique(restored_labels)) <= {0, 12}
assert np.count_nonzero(restored_labels == 12) > 0

# Prueba de continuidad 3D con IDs locales distintos por corte
synthetic_instances = np.zeros((3, 32, 32), dtype=np.uint16)
synthetic_semantics = np.zeros_like(synthetic_instances, dtype=np.uint8)
for z_index, (left_id, right_id, shift) in enumerate([(8, 4, 0), (2, 9, 1), (7, 3, 2)]):
    synthetic_instances[z_index, 4 + shift:10 + shift, 4:10] = left_id
    synthetic_semantics[z_index, 4 + shift:10 + shift, 4:10] = 1
    synthetic_instances[z_index, 20 - shift:26 - shift, 20:26] = right_id
    synthetic_semantics[z_index, 20 - shift:26 - shift, 20:26] = 3

linked_volume, linked_classes, per_slice_ids, association_events = link_slice_instances_to_volume(
    synthetic_instances, synthetic_semantics
)
left_tracks = [per_slice_ids[z][local_id] for z, local_id in enumerate([8, 2, 7])]
right_tracks = [per_slice_ids[z][local_id] for z, local_id in enumerate([4, 9, 3])]
assert len(set(left_tracks)) == 1 and len(set(right_tracks)) == 1
assert left_tracks[0] != right_tracks[0]
assert linked_classes[left_tracks[0]] == 1 and linked_classes[right_tracks[0]] == 3
assert not association_events
print("Letterbox restaurado:", restored_labels.shape, "| IDs conservados:", np.unique(restored_labels).tolist())
print("Asociación 3D:", left_tracks[0], right_tracks[0], "| alertas:", association_events)


<string>:42: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
<string>:47: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)


Letterbox restaurado: (40, 80) | IDs conservados: [0, 12]
Asociación 3D: 2 1 | alertas: []
